### Fashion-MNIST 不降维基准

流程：读取数据 → 复用 PCA / UMAP 结果里的抽样索引 → 保存原始 784 维数据。

这份结果用作对比基准：后面聚类时，PCA、UMAP 和原始数据用的是同一批 7000 个样本。

需要先运行 `fashion-PCA.ipynb` 和 `fashion-UMAP.ipynb`，生成 `result/` 下的两个降维结果文件。

In [1]:
import gzip
from pathlib import Path

import numpy as np

SEED = 42

#### 1. 读取数据

和两个降维 notebook 相同：训练集和测试集合并成 70000 个样本，每张图展平成 784 维，像素缩放到 [0, 1]。

In [2]:
DATA_DIR = Path('dataset')
FILES = ['train-images-idx3-ubyte.gz', 'train-labels-idx1-ubyte.gz',
         't10k-images-idx3-ubyte.gz', 't10k-labels-idx1-ubyte.gz']

def load_images(path):
    # idx3 头部 16 字节：magic, 数量, 行数, 列数
    with gzip.open(path, 'rb') as f:
        return np.frombuffer(f.read(), dtype=np.uint8, offset=16).reshape(-1, 784)

def load_labels(path):
    # idx1 头部 8 字节：magic, 数量
    with gzip.open(path, 'rb') as f:
        return np.frombuffer(f.read(), dtype=np.uint8, offset=8)

X_full = np.concatenate([load_images(DATA_DIR / FILES[0]),
                         load_images(DATA_DIR / FILES[2])]).astype(np.float32) / 255.0
y_full = np.concatenate([load_labels(DATA_DIR / FILES[1]),
                         load_labels(DATA_DIR / FILES[3])])

print('X_full:', X_full.shape, ' y_full:', y_full.shape)

X_full: (70000, 784)  y_full: (70000,)


#### 2. 复用分层抽样结果

不重新抽样，直接读 PCA 结果里的 `sample_idx`，并核对 UMAP 结果里的索引和标签与之完全相同。

In [3]:
with np.load(Path('result') / 'fashion-pca-redu-result.npz') as pca_r, \
     np.load(Path('result') / 'fashion-umap-redu-result.npz') as umap_r:
    sample_idx = pca_r['sample_idx']
    class_names = pca_r['class_names']
    assert np.array_equal(sample_idx, umap_r['sample_idx'])
    assert np.array_equal(pca_r['labels'], umap_r['labels'])
    assert np.array_equal(pca_r['labels'], y_full[sample_idx])

X = X_full[sample_idx]
y = y_full[sample_idx]

print('X:', X.shape, ' y:', y.shape)
print('前 10 个抽中的索引:', sample_idx[:10])
print('每类数量:', np.bincount(y).tolist())

X: (7000, 784)  y: (7000,)
前 10 个抽中的索引: [13 21 22 25 40 50 60 68 86 93]
每类数量: [700, 700, 700, 700, 700, 700, 700, 700, 700, 700]


#### 3. 保存结果

写入 `result/fashion-raw-result.npz`，键名与两个降维结果一致，后续聚类可以用同一段代码读取：

```python
r = np.load('result/fashion-raw-result.npz')
X, y = r['embedding'], r['labels']
```

| 键 | 内容 |
|---|---|
| `embedding` | (7000, 784) 原始像素，聚类用 |
| `labels` | 真实类别（整数），只用于 ARI、NMI 等外部指标，聚类时不能用 |
| `class_names` | 类别名，下标对应 `labels` |
| `sample_idx` | 在 70000 个原始样本中的索引，与 PCA、UMAP 结果相同 |
| `seed` | 随机种子 |

In [4]:
fashion_raw_result = {
    'embedding': X.astype(np.float32),
    'labels': y,
    'class_names': class_names,
    'sample_idx': sample_idx,
    'seed': SEED,
}

RESULT_FILE = Path('result') / 'fashion-raw-result.npz'
np.savez(RESULT_FILE, **fashion_raw_result)

# 读回来逐项核对
with np.load(RESULT_FILE) as saved:
    assert all(np.array_equal(saved[k], v) for k, v in fashion_raw_result.items())
print('已保存并校验:', RESULT_FILE)
for k, v in fashion_raw_result.items():
    print(f'  {k}: shape={np.shape(v)}, dtype={np.asarray(v).dtype}')

已保存并校验: result\fashion-raw-result.npz
  embedding: shape=(7000, 784), dtype=float32
  labels: shape=(7000,), dtype=uint8
  class_names: shape=(10,), dtype=<U11
  sample_idx: shape=(7000,), dtype=int64
  seed: shape=(), dtype=int64
